In [ ]:
!pip3 install pandas matplotlib

In [ ]:
import os
import pandas as pd
from opensearchpy import OpenSearch, Search

In [ ]:
index_name = 'my-org-index-2024-05-21--1'
endpoint = os.getenv("OPENSEARCH_ENDPOINT")
opensearch = OpenSearch(endpoint, verify_certs=False)

In [ ]:
#s = Search(using=opensearch, index=index_name)
#s.execute()

raw_results = opensearch.search(index=index_name, body={"size": 10000, "query": {"match_all": {}}})
doc_ids = list(map(lambda e: e["_id"], raw_results['hits']['hits']))
doc_ids[0]

In [ ]:
%%capture
term_vectors_flat = list(map(lambda e: opensearch.termvectors(index=index_name, id=e), doc_ids))

In [ ]:
term_vectors_flat

In [ ]:
terms_flat = list(map(lambda e: e['term_vectors']['thread_body']['terms'] if 'thread_body' in e['term_vectors'] else {}, term_vectors_flat))
terms_flat[0]

In [ ]:
corpus_terms = {}
for terms in terms_flat:
    for term in terms.keys():
        corpus_terms[term] = corpus_terms[term] + terms[term]['term_freq'] if term in corpus_terms else terms[term]['term_freq']
corpus_terms

In [ ]:
terms_table = pd.DataFrame(corpus_terms.items(), columns=["terms", "count"])
terms_table.head()

In [ ]:
terms_table[terms_table["count"] > 5].sort_values(by="count", ascending=False).head(30)